In [ ]:
# pip install packages that are not in Pyodide
%pip install ipympl==0.9.3
%pip install seaborn==0.12.2



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [63]:
import time
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
from sklearn.exceptions import ConvergenceWarning

import seaborn as sns


from ipywidgets import interact, IntSlider, Dropdown

from cycler import cycler
%matplotlib widget


# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
] 
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

# MLP Application: Predicting Pressure in a Water Network

 In this example, we demonstrate how a Multilayer Perceptron (MLP) can be used to estimate nodal pressures in a water distribution network, as illustrated in the simplified schematic below:

```{figure} https://files.mude.citg.tudelft.nl/WDSAsset1v1.png
---
align: center
---
Simplified scheme of a branched water distribution system
```








Pressure estimation is critical for ensuring stable and efficient operation of water networks. Traditionally, this is achieved using physically-based hydrodynamic models, which simulate fluid behavior across the system. However, such models can be computationally expensive, especially when used in tasks like optimization or real-time control in large networks.

To address this limitation, we use a data-driven alternative: a neural network trained on simulation results from the original physical model. This allows us to approximate the mapping from pipe diameters to nodal pressures, achieving high accuracy with significantly reduced computational cost.

As a case study, we use data from the BakRyan water distribution system as seen in the figure below, which consists of 58 pipes and 35 nodes.

```{figure} https://files.mude.citg.tudelft.nl/BAK.png
---
align: center
---
Numerical results for the BakRyan water distribution system
```


### Mathematical Model


We can express the relationship between pipe diameters and nodal pressures using a function $\phi$ parameterized by the neural network weights $\mathbf{W}$:

$$
\mathbf{y} = \phi(\mathbf{x}, \mathbf{W})
$$

Where:

- **$\mathbf{y}$**: output data (nodal pressures, units: *mwc*)
- **$\phi$**: represents the Neural Network
- **$\mathbf{x}$**: input data (pipe diameters, units: *m*)
- **$\mathbf{W}$**: parameters of the MLP (unitless)



Having pairs of input-output data (**diameters** $\mathbf{x}$ and **pressures** $\mathbf{y}$) the goal is to find the parameters $\mathbf{W}$ that best fit the data.

The neural network we train is a fully connected MLP, as sketched below.


```{figure} https://files.mude.citg.tudelft.nl/ANN_image2.png
---
align: center
---
Artificial neural network representation, with a zoomed-in view of how a single neuron works. For this notebook, we have 58 input features (pipe diameters) and 35 outputs (nodal pressures).
```




## Load data
we load the data and below you can check the dimensions of the *features* (the pipe diameters) and the *targets* (the nodal pressures).

In [54]:
import os
from urllib.request import urlretrieve

def findfile(fname):
    if not os.path.isfile(fname):
        print(f"Downloading {fname}...")
        urlretrieve('http://files.mude.citg.tudelft.nl/'+fname, fname)

findfile('targets_BAK.csv')
findfile('features_BAK.csv')

features = np.loadtxt('features_BAK.csv', delimiter=",")
targets = np.loadtxt('targets_BAK.csv', delimiter=",")


This page contains interactive python element: click {fa}`rocket` --> {guilabel}`Live Code` in the top right corner to activate it.

In [55]:
print('Dimensions of features (X):', features.shape)
print('Dimensions of targets  (t):', targets.shape)

Dimensions of features (X): (10000, 58)
Dimensions of targets  (t): (10000, 35)


## Training an MLP with scikit-learn


To train a neural network in Python, we can use scikit-learn's `MLPRegressor` class. This interface allows us to define and train a fully connected multilayer perceptron with with just a few lines of code.

We only need to specify the number of hidden layers and neurons, the activation function, and the number of training epochs. 
```python
from sklearn.neural_network import MLPRegressor

model = MLPRegressor(
    hidden_layer_sizes=(50, 50),   # Two hidden layers with 50 neurons each
    activation='relu',             # Activation function ('identity', 'logistic', 'tanh', 'relu')
    solver='sgd',                 # Optimizer 
    max_iter=100,                  # Number of epochs
    early_stopping=True,          # Stop if validation score does not improve
    validation_fraction=0.1,      # 10% of training data used for validation
    random_state=42
)
```








Once the model is defined, we call .fit() to train it:

```python

model.fit(X_train, t_train)

```

This single call can do all internal operations: shuffling, batching, gradient descent, and monitoring validation performance.

During training, scikit-learn records the training loss at each epoch, which we can access via `model.loss_curve_.` We can use `model.validation_scores_.` to get the $R^2$ score, as a way of recording the validation loss.

These can be plotted to analyze model convergence. You can play your self with some of the hyperparamters and see how they effect the training and validation plot below.

 **Be aware that the required computations can take a few moments to run.** 

In [ ]:


def prepare_data(features, targets):
    """
    Splits and normalizes the dataset.

    Returns:
        X_train, X_val, t_train, t_val: Scaled training and validation sets.
    """

    X_train, X_val_test, t_train, t_val_test = train_test_split(
        features, targets, test_size=0.20, random_state=42, shuffle=True
    )

    X_val, _, t_val, _ = train_test_split(
        X_val_test, t_val_test, test_size=0.50, random_state=42, shuffle=True
    )

    scaler_X = MinMaxScaler().fit(X_train)
    scaler_t = MinMaxScaler().fit(t_train)

    X_train_n = scaler_X.transform(X_train)
    X_val_n   = scaler_X.transform(X_val)
    t_train_n = scaler_t.transform(t_train)
    t_val_n   = scaler_t.transform(t_val)

    return X_train_n, X_val_n, t_train_n, t_val_n


warnings.filterwarnings("ignore", category=ConvergenceWarning)

def interactive_train_plot(features, targets):
    def run(num_layers=2, neurons=50, activation="tanh", n_epochs=30):
        X_train, X_val, t_train, t_val = prepare_data(features, targets)
        hidden = tuple([neurons] * num_layers)

        model = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation=activation,
            solver="sgd",  
            learning_rate_init=1e-3,
            batch_size=64,
            max_iter=n_epochs,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=42,
        )

        model.fit(X_train, t_train)

        # Combined figure with two subplots
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Training Loss (log scale)
        if hasattr(model, "loss_curve_"):
            axes[0].plot(model.loss_curve_, color='blue', linewidth=2, label="Training Loss")
            axes[0].set_yscale("log")
            axes[0].set_title("Training Loss Curve")
            axes[0].set_xlabel("Epochs")
            axes[0].set_ylabel("Loss (MSE)")
            axes[0].grid(True, linestyle="--", alpha=0.5)
            axes[0].legend()

        # Validation R²
        val_scores = getattr(model, "validation_scores_", None)
        if val_scores is not None and len(val_scores) > 0:
            axes[1].plot(val_scores, color='orange', linewidth=2, label="Validation $R^2$")
            axes[1].set_title("Validation Score Curve")
            axes[1].set_xlabel("Epochs")
            axes[1].set_ylabel("$R^2$ Score")
            axes[1].grid(True, linestyle="--", alpha=0.5)
            axes[1].legend()

        fig.suptitle("MLP Training Progress", fontsize=16, fontweight="bold")
        fig.tight_layout()
        plt.show()

    interact(
        run,
        num_layers=IntSlider(min=1, max=10, step=1, value=2, description="Layers"),
        neurons=IntSlider(min=10, max=100, step=5, value=50, description="Neurons"),
        n_epochs=IntSlider(min=10, max=100, step=1, value=30, description="Epochs"),
        activation=Dropdown(
            options=["identity", "logistic", "tanh", "relu"],
            value="tanh",
            description="Activation"
        ),
    )


In [62]:
interactive_train_plot(features, targets)



interactive(children=(IntSlider(value=2, description='Layers', max=10, min=1), IntSlider(value=50, description…